# Minervini 트렌드 템플릿 + RS 랭킹 분석

`universe_2015_analysis.ipynb`에서 생성한 데이터를 Google Drive에서 불러와 분석합니다.

## 입력 파일 (Drive `MyDrive/universe_2015_data/`)
- `universe_ohlcv.parquet` — 164종목 OHLCV
- `kospi_1001.parquet` — KOSPI 지수
- `vix.parquet` — VIX
- `source_log.csv` — 소스 로그

## 분석 내용
1. **파생 지표 계산**: 50/150/200일 SMA, 52주 고저, 수익률
2. **Minervini 트렌드 템플릿** 8개 조건 통과 종목
3. **RS (상대강도) 랭킹** — KOSPI 대비 12개월 가중 수익률
4. **VIX 게이트** 구간별 시장 상태 검증

## 1. Drive 마운트 & 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR = Path('/content/drive/MyDrive/universe_2015_data')
assert DATA_DIR.exists(), f'데이터 폴더 없음: {DATA_DIR}'

panel = pd.read_parquet(DATA_DIR / 'universe_ohlcv.parquet')
kospi = pd.read_parquet(DATA_DIR / 'kospi_1001.parquet')
vix   = pd.read_parquet(DATA_DIR / 'vix.parquet')
source_log = pd.read_csv(DATA_DIR / 'source_log.csv', dtype={'ticker': str})

NAME_MAP = dict(zip(source_log['ticker'], source_log['name']))
TICKERS = panel.columns.get_level_values('ticker').unique().tolist()

print(f'유니버스: {len(TICKERS)}종목')
print(f'기간    : {panel.index[0].date()} ~ {panel.index[-1].date()} ({len(panel):,} 거래일)')
print(f'KOSPI   : {len(kospi):,} rows')
print(f'VIX     : {len(vix):,} rows')

## 2. 종가·거래량 매트릭스 추출

MultiIndex panel을 field별로 분리 (행=날짜, 열=종목).

In [ ]:
close  = panel.xs('Close',  level='field', axis=1).sort_index()
high   = panel.xs('High',   level='field', axis=1).sort_index()
low    = panel.xs('Low',    level='field', axis=1).sort_index()
volume = panel.xs('Volume', level='field', axis=1).sort_index()

print(f'close shape: {close.shape}  (거래일 x 종목)')
close.tail(3)

## 3. 파생 지표 — SMA·52주 고저·수익률

In [ ]:
sma50  = close.rolling(50,  min_periods=50).mean()
sma150 = close.rolling(150, min_periods=150).mean()
sma200 = close.rolling(200, min_periods=200).mean()

high_52w = close.rolling(252, min_periods=252).max()
low_52w  = close.rolling(252, min_periods=252).min()

# 200일 MA가 최근 22거래일(약 1개월)간 상승 추세인지
sma200_trending_up = sma200 > sma200.shift(22)

print('파생 지표 계산 완료')

## 4. Minervini 트렌드 템플릿 (8개 조건)

Mark Minervini, *Trade Like a Stock Market Wizard* 원전 기준:
1. 종가 > SMA150, SMA200
2. SMA150 > SMA200
3. SMA200 최소 1개월간 상승 추세
4. SMA50 > SMA150, SMA200
5. 종가 > SMA50
6. 종가 ≥ 52주 저점 × 1.30 (30% 이상 반등)
7. 종가 ≥ 52주 고점 × 0.75 (25% 이내 근접)
8. RS 랭킹 ≥ 70 (별도 계산, 다음 섹션)

In [ ]:
def minervini_template(as_of=None):
    """지정 날짜(기본: 최근 거래일) 기준 트렌드 템플릿 통과 여부 DataFrame 반환."""
    idx = close.index[-1] if as_of is None else pd.Timestamp(as_of)

    row = pd.DataFrame({
        'close': close.loc[idx],
        'sma50': sma50.loc[idx],
        'sma150': sma150.loc[idx],
        'sma200': sma200.loc[idx],
        'high_52w': high_52w.loc[idx],
        'low_52w': low_52w.loc[idx],
        'sma200_up': sma200_trending_up.loc[idx],
    })

    row['c1_above_150_200'] = (row['close'] > row['sma150']) & (row['close'] > row['sma200'])
    row['c2_150_above_200'] = row['sma150'] > row['sma200']
    row['c3_200_trending_up'] = row['sma200_up']
    row['c4_50_above_150_200'] = (row['sma50'] > row['sma150']) & (row['sma50'] > row['sma200'])
    row['c5_close_above_50'] = row['close'] > row['sma50']
    row['c6_above_30pct_low'] = row['close'] >= row['low_52w'] * 1.30
    row['c7_near_52w_high'] = row['close'] >= row['high_52w'] * 0.75

    cond_cols = [c for c in row.columns if c.startswith('c')]
    row['pass_count'] = row[cond_cols].sum(axis=1)
    row['name'] = [NAME_MAP.get(t, '?') for t in row.index]

    return row, idx


result, as_of = minervini_template()
print(f'기준일: {as_of.date()}')

# 7개 조건 모두 통과 (RS 조건 제외, 7/7)
passed = result[result['pass_count'] == 7].sort_values('close', ascending=False)
print(f'7개 조건 전부 통과: {len(passed)}종목')
passed[['name', 'close', 'sma50', 'sma200', 'high_52w']].head(30)

In [ ]:
# 거의 통과 (6/7) — 어떤 조건 하나 걸리는지 확인
almost = result[result['pass_count'] == 6].copy()
cond_cols = [c for c in result.columns if c.startswith('c')]
almost['missing'] = almost[cond_cols].apply(
    lambda r: ', '.join([c for c in cond_cols if not r[c]]), axis=1
)
print(f'6/7 통과 (한 조건만 미달): {len(almost)}종목')
almost[['name', 'close', 'missing']].sort_values('missing')

## 5. RS (Relative Strength) 랭킹

IBD 방식: 12개월 수익률 가중 평균 (최근 분기 비중 높게). 모든 유니버스 내 percentile로 환산.

In [ ]:
def rs_score(prices: pd.DataFrame, as_of=None):
    """IBD 스타일 RS: 2*Q1 + Q2 + Q3 + Q4 (Q1=최근 분기)."""
    idx = prices.index[-1] if as_of is None else pd.Timestamp(as_of)
    pos = prices.index.get_loc(idx)

    def ret(offset_days):
        if pos - offset_days < 0:
            return pd.Series(np.nan, index=prices.columns)
        return prices.iloc[pos] / prices.iloc[pos - offset_days] - 1

    q1 = ret(63)    # 최근 3개월
    q2 = ret(126)   # 6개월
    q3 = ret(189)   # 9개월
    q4 = ret(252)   # 12개월

    raw = 2 * q1 + q2 + q3 + q4
    # percentile 랭킹 (1~99)
    rank = raw.rank(pct=True) * 99
    return pd.DataFrame({'raw_rs': raw, 'rs_rank': rank})


rs = rs_score(close)
rs['name'] = [NAME_MAP.get(t, '?') for t in rs.index]
rs = rs.sort_values('rs_rank', ascending=False)

print(f'RS Top 20 (기준일 {close.index[-1].date()})')
rs.head(20)

## 6. 완전 통과 — Minervini 7/7 + RS ≥ 70

In [ ]:
combined = result.join(rs[['rs_rank']])
combined['c8_rs_70+'] = combined['rs_rank'] >= 70
combined['total_pass'] = combined['pass_count'] + combined['c8_rs_70+'].astype(int)

winners = combined[combined['total_pass'] == 8].sort_values('rs_rank', ascending=False)
print(f'Minervini 8/8 통과: {len(winners)}종목')
winners[['name', 'close', 'sma50', 'sma200', 'high_52w', 'rs_rank']]

## 7. VIX 게이트 분석

VIX 구간별 KOSPI 20거래일 후행 수익률 분포 — 고변동 시장 리스크 검증.

In [ ]:
kospi_close = kospi['Close']
kospi_fwd20 = kospi_close.shift(-20) / kospi_close - 1

vix_close = vix['Close'].reindex(kospi_close.index, method='ffill')

bins = [0, 15, 20, 25, 30, 100]
labels = ['< 15 (안정)', '15-20', '20-25 (주의)', '25-30 (경계)', '30+ (위험)']
vix_bucket = pd.cut(vix_close, bins=bins, labels=labels)

stats = pd.DataFrame({
    'vix_bucket': vix_bucket,
    'fwd20_ret': kospi_fwd20,
}).dropna().groupby('vix_bucket')['fwd20_ret'].agg(['count', 'mean', 'median', 'std'])
stats['hit_rate'] = pd.DataFrame({
    'vix_bucket': vix_bucket,
    'fwd20_ret': kospi_fwd20,
}).dropna().groupby('vix_bucket')['fwd20_ret'].apply(lambda s: (s > 0).mean())

print('VIX 구간별 KOSPI 20일 후행 수익률:')
stats[['count', 'mean', 'median', 'hit_rate']].style.format({
    'mean': '{:.2%}', 'median': '{:.2%}', 'hit_rate': '{:.1%}',
})

## 8. 시각화

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# KOSPI + MA60
ax1 = axes[0]
kospi_close.plot(ax=ax1, label='KOSPI', lw=1)
kospi_close.rolling(60).mean().plot(ax=ax1, label='MA60', lw=1, alpha=0.7)
ax1.set_title('KOSPI & MA60')
ax1.legend()
ax1.grid(alpha=0.3)

# VIX
ax2 = axes[1]
vix_close.plot(ax=ax2, label='VIX', color='crimson', lw=1)
ax2.axhline(20, color='orange', ls='--', lw=0.8, label='20 (주의)')
ax2.axhline(30, color='red', ls='--', lw=0.8, label='30 (위험)')
ax2.set_title('VIX')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Minervini 통과 종목의 가격·MA 차트 (최대 6종목)
to_plot = winners.head(6)
if len(to_plot) == 0:
    to_plot = combined.sort_values('total_pass', ascending=False).head(6)
    print('통과 종목 없음 — 상위 6종목으로 대체')

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (ticker, _) in zip(axes.flat, to_plot.iterrows()):
    name = NAME_MAP.get(ticker, ticker)
    s = close[ticker].dropna().iloc[-500:]  # 최근 2년
    s.plot(ax=ax, lw=1, label='Close')
    sma50[ticker].dropna().iloc[-500:].plot(ax=ax, lw=0.8, label='SMA50', alpha=0.7)
    sma150[ticker].dropna().iloc[-500:].plot(ax=ax, lw=0.8, label='SMA150', alpha=0.7)
    sma200[ticker].dropna().iloc[-500:].plot(ax=ax, lw=0.8, label='SMA200', alpha=0.7)
    ax.set_title(f'{ticker} {name}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 결과 저장 (Drive)

In [ ]:
OUTPUT = DATA_DIR / 'analysis_output'
OUTPUT.mkdir(exist_ok=True)

combined.to_csv(OUTPUT / f'minervini_{as_of.date()}.csv', encoding='utf-8-sig')
stats.to_csv(OUTPUT / f'vix_gate_stats_{as_of.date()}.csv', encoding='utf-8-sig')

print('저장 완료:')
for p in sorted(OUTPUT.iterdir()):
    print(f'  {p.relative_to(DATA_DIR)}')